# 🚀 SUSA-1B CodeMind — Official Google Colab GPU Pretraining

> **An original coding-focused Transformer language model trained from scratch under the SUSA ecosystem.**

---

### 📌 Guidelines & Rules for Colab Training:
1. **Zero Compute Cost:** Uses Google Colab Free Tier GPU (e.g. NVIDIA T4 / V100).
2. **Real Weights from Scratch:** Randomly initialized weights. No repurposed external weights.
3. **Persistent Checkpointing:** Active checkpoints save to local Colab NVMe storage for maximum I/O throughput, and sync to Google Drive for disconnection recovery.
4. **Git-Centric Source of Truth:** All modules, configs, and tests live in the repository.

## Step 1: Connect to GitHub & Clone Repository

In [ ]:
import os
import sys

# Enter your GitHub repository URL
REPO_URL = input("Enter your GitHub repository URL (e.g., https://github.com/saketh-nandu/susa-1b-codemind.git): ").strip()
REPO_NAME = "susa-1b-codemind"

if os.path.exists(REPO_NAME):
    print(f"Pulling latest approved commit in {REPO_NAME}...")
    !cd {REPO_NAME} && git pull origin main
else:
    print(f"Cloning {REPO_URL} into Colab runtime...")
    !git clone {REPO_URL} {REPO_NAME}

%cd {REPO_NAME}
!pwd && git log -n 1 --oneline

## Step 2: Install Documented Dependencies

In [ ]:
# Install project in editable mode with colab and dev dependencies
!pip install --upgrade pip
!pip install -e ".[colab,dev]"
!pip install psutil pyyaml tokenizers

## Step 3: Inspect GPU Hardware & Environment Diagnostics

In [ ]:
import torch
from susacodemind.utils.system_checker import SystemInfo

sys_info = SystemInfo.inspect()
print(sys_info.summary())

if not sys_info.cuda_available:
    print("\n" + "!" * 68)
    print("  [CRITICAL WARNING] No CUDA GPU detected in this Colab session!")
    print("  Please change runtime accelerator:")
    print("  1. Go to top menu: Runtime -> Change runtime type")
    print("  2. Hardware accelerator: Select 'T4 GPU'")
    print("  3. Save and re-run this notebook.")
    print("!" * 68)
    # Stop execution before expensive CPU fallback on large models
    raise SystemExit("Execution paused: GPU required for training beyond smoke test.")
else:
    print(f"\n[OK] CUDA GPU ready: {torch.cuda.get_device_name(0)} ({sys_info.gpu_devices[0]['total_vram_gb']} GiB VRAM)")

## Step 4: Mount Google Drive (For Persistent Checkpoints)

In [ ]:
# Authorize Google Drive via standard Colab flow
ENABLE_DRIVE_SYNC = True #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/SUSA_CodeMind_Checkpoints"

if ENABLE_DRIVE_SYNC:
    from google.colab import drive
    print("Authorizing Google Drive mount...")
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Google Drive persistent directory ready at: {DRIVE_DIR}")
else:
    print("Google Drive sync disabled. Checkpoints will remain in local runtime ephemeral storage.")

## Step 5: Select Profile & Validate Resource Feasibility

In [ ]:
# Select model configuration profile
PROFILE = "small" #@param ["smoke", "small", "medium", "target_1b"]

from susacodemind.model.config import ModelConfig
from susacodemind.utils.resource_estimator import estimate_resources

cfg_path = f"configs/{PROFILE}.yaml"
model_cfg = ModelConfig.from_yaml(cfg_path)
print(model_cfg.print_parameter_summary())

# Check memory feasibility against Colab VRAM
res = estimate_resources(model_cfg, batch_size=2, seq_len=model_cfg.max_seq_len)
print(res.summary())

if PROFILE == "target_1b":
    print("\n[NOTICE] Profile 'target_1b' (~1B parameters) requires gradient checkpointing and micro-batching on free T4.")
    print("For rapid initial experiments, 'small' (~34M) or 'medium' (~222M) is recommended.")

## Step 6: Run Comprehensive Test Suite

In [ ]:
# Verify all components pass unit and regression tests
!pytest tests/ -v

## Step 7: Train Tokenizer (If Not Available)

In [ ]:
TOKENIZER_DIR = f"tokenizer_cache/{PROFILE}"
VOCAB_SIZE = model_cfg.vocab_size

# Train or load Byte-Level BPE tokenizer
if not os.path.exists(os.path.join(TOKENIZER_DIR, "tokenizer.json")):
    print(f"Training tokenizer for profile {PROFILE} with vocab size {VOCAB_SIZE:,}...")
    !python scripts/train_tokenizer.py --use-synthetic --vocab-size {VOCAB_SIZE} --output-dir {TOKENIZER_DIR}
else:
    print(f"Tokenizer already exists at {TOKENIZER_DIR}.")

## Step 8: Prepare Sharded Binary Dataset

In [ ]:
# Preprocess and stream dataset to zero-RAM memory-mapped binary shards
if not os.path.exists("data/train.bin"):
    print("Cleaning, scrubbing secrets, and creating binary dataset shards...")
    !python scripts/prepare_dataset.py --tokenizer-dir {TOKENIZER_DIR} --use-synthetic --output-dir data
else:
    print("Binary dataset shards already exist in data/")

## Step 9: Launch Pretraining with Real-Time Metrics & Checkpoints

In [ ]:
TRAINING_STEPS = 50 #@param {type:"integer"}
RESUME_CHECKPOINT = "" #@param {type:"string"}

train_cmd = f"python scripts/train.py --config configs/{PROFILE}.yaml --train-bin data/train.bin --val-bin data/val.bin --max-steps {TRAINING_STEPS}"
if RESUME_CHECKPOINT.strip():
    train_cmd += f" --resume-from {RESUME_CHECKPOINT.strip()}"

print(f"Executing: {train_cmd}")
!{train_cmd}

## Step 10: Sync Checkpoints to Google Drive & Export Run Manifest

In [ ]:
if ENABLE_DRIVE_SYNC and os.path.exists("/content/drive/MyDrive"):
    print(f"Syncing checkpoints to Google Drive: {DRIVE_DIR}...")
    !python scripts/sync_checkpoints.py --src-dir checkpoints --dest-dir "{DRIVE_DIR}"
else:
    print("Skipping Drive synchronization (Drive not mounted).")

## Step 11: Final Training Summary & Health Check

In [ ]:
import glob

saved_ckpts = sorted(glob.glob("checkpoints/*.pt"))
print("=" * 70)
print("  SUSA-1B CODEMIND — EXECUTION SUMMARY")
print("=" * 70)
print(f"  Profile:              {PROFILE}")
print(f"  Model Parameters:     {model_cfg.calculate_exact_parameters()['formatted_total']}")
print(f"  Local Checkpoints:    {len(saved_ckpts)} saved ({saved_ckpts})")
if ENABLE_DRIVE_SYNC:
    print(f"  Google Drive Backup:  {DRIVE_DIR}")
print("  Hardware Cost:        ₹0 (Free Colab Session)")
print("=" * 70)
print("Pretraining iteration complete!")